# Objective:
# Build item-based recommendation system using cosine similarity

In [2]:
import pandas as pd



In [3]:
df = pd.read_excel('../data/Online_Retail.xlsx')

In [4]:
df = df.dropna(subset=['CustomerID'])
df = df.dropna(subset=['Description'])
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df = df.drop_duplicates()

df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

In [5]:
customer_product = df.pivot_table(
    index='CustomerID',
    columns='Description',
    values='Quantity',
    aggfunc='sum',
    fill_value=0
)

In [6]:
customer_product

Description,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
CustomerID,,,,,,,,,,,,,,,,,,,,,
12346.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12347.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12348.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12349.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12350.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
18281.0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
18282.0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
from sklearn.metrics.pairwise import cosine_similarity


In [8]:

similarity = cosine_similarity(customer_product.T)

In [9]:
similarity_df = pd.DataFrame(
    similarity,
    index=customer_product.columns,
    columns=customer_product.columns
)

In [10]:
def recommend_products(product_name, top_n=5):
    similar_products = similarity_df[product_name].sort_values(ascending=False)
    return similar_products[1:top_n+1]

In [11]:
recommend_products("WHITE HANGING HEART T-LIGHT HOLDER")

Description
GIN + TONIC DIET METAL SIGN         0.750192
RED HANGING HEART T-LIGHT HOLDER    0.658714
WASHROOM METAL SIGN                 0.643520
LAUNDRY 15C METAL SIGN              0.642200
GREEN VINTAGE SPOT BEAKER           0.631463
Name: WHITE HANGING HEART T-LIGHT HOLDER, dtype: float64

In [14]:
def recommend_for_customer(customer_id):
    customer_data = customer_product.loc[customer_id]
    bought_products = customer_data[customer_data > 0].index

    recommendations = {}

    for product in bought_products:
        similar = similarity_df[product].sort_values(ascending=False)[1:4]
        for item in similar.index:
            if item not in bought_products:
                recommendations[item] = recommendations.get(item, 0) + similar[item]

    return [(item, float(score)) for item, score in sorted(recommendations.items(), key=lambda x: x[1], reverse=True)[:5]]

In [15]:
recommend_for_customer(12347)

[('ROUND SNACK BOXES SET OF4 WOODLAND ', 4.115165848141077),
 ('PACK OF 72 SKULL CAKE CASES', 2.686619488468952),
 (' DOLLY GIRL BEAKER', 2.529743435735489),
 ('5 HOOK HANGER RED MAGIC TOADSTOOL', 2.505174363045294),
 ('MAGIC DRAWING SLATE DOLLY GIRL ', 2.477977355577449)]